# Battery Surrogate Model Launcher

Unified notebook for training and evaluating battery thermal surrogate models.

**Supported Models:**
- **MLP Pointwise**: Feed-forward network predicting (T, bc_V) from per-row features
- **Recurrent**: GRU/LSTM sequence model with history-lag autoregressive rollout

```mermaid
flowchart LR
    A[Raw OPs] --> B[OP Bundle]
    B --> C{Model Type}
    C -->|mlp_pointwise| D[Pointwise Dataset]
    C -->|recurrent| E[Sequence Dataset]
    D --> F[MLP Trainer]
    E --> G[Sequence Trainer]
    F --> H[Checkpoint + Normalizer]
    G --> H
    H --> I[Evaluation]
```

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Add src to path for imports
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import torch
import numpy as np
import yaml
import matplotlib.pyplot as plt

# Battery surrogate imports
from battery_surrogate.cli.train import train_from_config
from battery_surrogate.model.registry import build_model, build_datasets
from battery_surrogate.model.normalizer import PointwiseNormalizer
from battery_surrogate.model.split import resolve_split, validate_coverage
from battery_surrogate.model.evaluate import evaluate_on_ops
from battery_surrogate.model.evaluate_sequence import (
    evaluate_sequence_model,
    history_length_benchmark,
    plot_error_curves,
)
from battery_surrogate.data.loader import load_op

# Device setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"Working directory: {ROOT}")

## 2. Model Selection & Configuration

Choose model type and load/customize configuration.

In [ ]:
# ============================================================
# MODEL SELECTION — Change this to switch between models
# ============================================================
MODEL_TYPE = "mlp_pointwise"  # Options: "mlp_pointwise" or "recurrent"

# Load base config
CONFIG_PATH = ROOT / "configs" / "model" / f"{MODEL_TYPE}.yaml"
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

print(f"Loaded config: {CONFIG_PATH}")
print(f"Model type: {config['model']['type']}")

In [ ]:
# ============================================================
# OPTIONAL: Inline config overrides
# ============================================================

# Override OP split (uncomment to customize)
# config["data"]["train_ops"] = ["OP01", "OP02", "OP08"]
# config["data"]["val_ops"] = ["OP09"]
# config["data"]["test_ops"] = ["OP10", "OP11"]

# Override training params
# config["train"]["epochs"] = 10
# config["train"]["batch_size"] = 2048

# Override subsample (faster training, less data)
# config["data"]["subsample_time"] = 100

# Recurrent-specific: history length
if MODEL_TYPE == "recurrent":
    config["model"].setdefault("history_length", 8)
    print(f"History length (k): {config['model']['history_length']}")

# Show final config
print("\n=== Final Configuration ===")
print(yaml.dump(config, default_flow_style=False))

## 3. Data Preview & Coverage Validation

In [ ]:
# Resolve OP split
split = resolve_split(config)

print("=== OP Split ===")
print(f"Train OPs ({len(split['train'])}): {split['train']}")
print(f"Val OPs   ({len(split['val'])}):   {split['val']}")
print(f"Test OPs  ({len(split['test'])}):  {split['test']}")

In [ ]:
# Preview one bundle
sample_op = split["train"][0]
bundle = load_op(sample_op)

print(f"\n=== Sample Bundle: {sample_op} ===")
print(f"Time steps: {bundle.t_fast.shape[0]}")
print(f"Sensors: {bundle.xyz.shape[0]}")
print(f"T shape: {bundle.T.shape}")
print(f"bc_V shape: {bundle.bc_V.shape}")
print(f"Time range: {bundle.t_fast[0]:.2f}s to {bundle.t_fast[-1]:.2f}s")

## 4. Training (In-Process)

Launch training using the unified `train_from_config()` dispatcher.

In [ ]:
# ============================================================
# TRAIN THE MODEL (in-process, recommended)
# ============================================================

print(f"Training {MODEL_TYPE} model...")
print(f"Epochs: {config['train']['epochs']}")
print(f"Batch size: {config['train']['batch_size']}")
print()

summary = train_from_config(config)

print("\n=== Training Complete ===")
print(f"Best val loss: {summary['best_val_loss']:.6f}")
print(f"Checkpoint dir: {summary['ckpt_dir']}")
print(f"Parameters: {summary.get('n_parameters', 'N/A'):,}")

### 4b. Optional: Shell-Out Training

Alternative: run training via CLI (useful for background jobs or HPC).

In [ ]:
# ============================================================
# OPTIONAL: Shell-out training (uncomment to use)
# ============================================================

# import subprocess
# result = subprocess.run(
#     ["python", "-m", "battery_surrogate.cli.train", "--config", str(CONFIG_PATH)],
#     cwd=str(ROOT),
#     capture_output=True,
#     text=True,
#     env={**os.environ, "PYTHONPATH": str(SRC)},
# )
# print(result.stdout)
# if result.returncode != 0:
#     print("STDERR:", result.stderr)

## 5. Load Artifacts & Evaluate

In [ ]:
# Load checkpoint directory from training summary
ckpt_dir = Path(summary["ckpt_dir"])

# Load normalizer
normalizer = PointwiseNormalizer.load(ckpt_dir / "normalizer.json")
print(f"Loaded normalizer from {ckpt_dir / 'normalizer.json'}")

# Load config
with open(ckpt_dir / "config.yaml", "r") as f:
    saved_config = yaml.safe_load(f)

# Rebuild model architecture
model = build_model(saved_config, n_sensors=363, seed=saved_config.get("seed", 42))

# Load weights
ckpt_path = ckpt_dir / "best.pt"
state_dict = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()

print(f"\nLoaded model from {ckpt_path}")
print(f"Model type: {saved_config['model']['type']}")
print(f"Parameters: {model.n_parameters:,}")

In [ ]:
# Evaluate on test OPs
test_ops = split["test"]
print(f"Evaluating on test OPs: {test_ops}")

if MODEL_TYPE == "mlp_pointwise":
    # MLP evaluation
    metrics = evaluate_on_ops(
        model,
        test_ops,
        normalizer,
        subsample_time=saved_config["data"].get("subsample_time", 1),
        ts_extrapolation=saved_config["data"].get("ts_extrapolation", "clamp"),
        device=DEVICE,
    )
else:
    # Recurrent evaluation (autoregressive rollout)
    metrics = evaluate_sequence_model(
        model,
        test_ops,
        normalizer,
        saved_config,
        device=DEVICE,
        max_sensors=50,  # Limit for faster eval
    )

print("\n=== Test Metrics ===")
print(f"Temperature (T):")
print(f"  MAE:  {metrics['mae_T']:.4f}")
print(f"  MSE:  {metrics['mse_T']:.6f}")
print(f"  R²:   {metrics['r2_T']:.4f}")
print(f"\nVoltage (bc_V):")
print(f"  MAE:  {metrics['mae_bc_V']:.4f}")
print(f"  MSE:  {metrics['mse_bc_V']:.6f}")
print(f"  R²:   {metrics['r2_bc_V']:.4f}")

## 6. Visualization

In [ ]:
# Plot predictions vs ground truth for one OP
sample_test_op = test_ops[0]
bundle = load_op(sample_test_op)

# Get predictions for a single sensor
sensor_idx = 0
subsample = saved_config["data"].get("subsample_time", 1)

if MODEL_TYPE == "mlp_pointwise":
    from battery_surrogate.model.features_pointwise import iter_pointwise_blocks
    
    time_indices = np.arange(0, bundle.t_fast.shape[0], subsample)
    y_true_all = []
    y_pred_all = []
    
    with torch.no_grad():
        for x_block, y_block, sensor_ids, time_idx in iter_pointwise_blocks(
            bundle, time_indices, ts_extrapolation="clamp"
        ):
            # Find rows for sensor_idx
            mask = sensor_ids == sensor_idx
            if mask.sum() == 0:
                continue
            x_norm = normalizer.transform_X(x_block[mask])
            x_tensor = torch.from_numpy(x_norm).to(DEVICE)
            pred_norm = model(x_tensor).cpu().numpy()
            pred = normalizer.inverse_Y(pred_norm)
            y_true_all.append(y_block[mask])
            y_pred_all.append(pred)
    
    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)
    t_vals = bundle.t_fast[time_indices]
else:
    from battery_surrogate.model.features_sequence import build_sequence_for_sensor
    
    features, targets, seq_len = build_sequence_for_sensor(
        bundle, sensor_idx, saved_config, "clamp"
    )
    features_norm = normalizer.transform_X(features)
    targets_norm = normalizer.transform_Y(targets)
    
    y0 = torch.from_numpy(targets_norm[0]).float().to(DEVICE)
    features_tensor = torch.from_numpy(features_norm).float().to(DEVICE)
    
    with torch.no_grad():
        preds_norm = model.rollout(features_tensor, y0).cpu().numpy()
    
    y_pred = normalizer.inverse_Y(preds_norm)
    y_true = targets
    t_vals = bundle.t_fast[::subsample][:seq_len]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(t_vals, y_true[:, 0], 'b-', label='Ground Truth', alpha=0.7)
axes[0].plot(t_vals, y_pred[:, 0], 'r--', label='Prediction', alpha=0.7)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Temperature (T)')
axes[0].set_title(f'{sample_test_op} - Sensor {sensor_idx} - Temperature')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_vals, y_true[:, 1], 'b-', label='Ground Truth', alpha=0.7)
axes[1].plot(t_vals, y_pred[:, 1], 'r--', label='Prediction', alpha=0.7)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Voltage (bc_V)')
axes[1].set_title(f'{sample_test_op} - Sensor {sensor_idx} - Voltage')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Error-vs-time curves (recurrent only)
if MODEL_TYPE == "recurrent" and "error_curves" in metrics:
    fig = plot_error_curves(metrics["error_curves"], sample_test_op, sensor_ids=[0, 1, 2])
    plt.show()

## 7. History-Length Benchmark (Recurrent Only)

Sweep different history lengths (k) to find the accuracy-cost sweet spot.

In [ ]:
# Run history-length benchmark (recurrent only)
if MODEL_TYPE == "recurrent":
    print("Running history-length benchmark...")
    print("(This may take several minutes)\n")
    
    # Benchmark with small k values and 1 epoch for speed
    benchmark_config = saved_config.copy()
    benchmark_config["train"]["epochs"] = 1
    benchmark_config["data"]["subsample_time"] = 100  # Fast sweep
    
    benchmark_df = history_length_benchmark(
        benchmark_config,
        k_values=[1, 2, 4, 8, 16],
        epochs_per_k=1,
        max_sensors=5,
        device=DEVICE,
    )
    
    print("=== History Length Benchmark Results ===")
    display(benchmark_df)
else:
    print("History-length benchmark is only applicable for recurrent models.")

In [ ]:
# Plot benchmark results
if MODEL_TYPE == "recurrent" and 'benchmark_df' in dir():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Accuracy vs k
    axes[0].plot(benchmark_df['k'], benchmark_df['r2_T'], 'bo-', label='R² (T)')
    axes[0].plot(benchmark_df['k'], benchmark_df['r2_bc_V'], 'ro-', label='R² (bc_V)')
    if 'recommended' in benchmark_df.columns:
        rec = benchmark_df[benchmark_df['recommended']]
        if not rec.empty:
            axes[0].axvline(rec['k'].values[0], color='g', linestyle='--', label='Recommended')
    axes[0].set_xlabel('History Length (k)')
    axes[0].set_ylabel('R²')
    axes[0].set_title('Accuracy vs History Length')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Lookback seconds vs k
    axes[1].plot(benchmark_df['k'], benchmark_df['lookback_seconds_median'], 'go-')
    axes[1].fill_between(
        benchmark_df['k'],
        benchmark_df['lookback_seconds_min'],
        benchmark_df['lookback_seconds_max'],
        alpha=0.2,
    )
    axes[1].set_xlabel('History Length (k)')
    axes[1].set_ylabel('Lookback Window (seconds)')
    axes[1].set_title('Lookback Window vs History Length')
    axes[1].grid(True, alpha=0.3)
    
    # Training time vs k
    axes[2].bar(benchmark_df['k'], benchmark_df['train_time_s'], color='purple', alpha=0.7)
    axes[2].set_xlabel('History Length (k)')
    axes[2].set_ylabel('Training Time (s)')
    axes[2].set_title('Training Cost vs History Length')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. Summary

Training and evaluation complete! Key outputs:

1. **Checkpoint**: `{ckpt_dir}/best.pt`
2. **Normalizer**: `{ckpt_dir}/normalizer.json`
3. **Config**: `{ckpt_dir}/config.yaml`

### Next Steps

- **Hyperparameter tuning**: Use Optuna integration (see `optuna_mlp_pointwise.py`)
- **Per-group normalization**: Enable `preprocess` block in config
- **Production deployment**: Export model for inference

In [ ]:
# Final summary
print("=" * 60)
print("TRAINING & EVALUATION SUMMARY")
print("=" * 60)
print(f"Model type:     {MODEL_TYPE}")
print(f"Best val loss:  {summary['best_val_loss']:.6f}")
print(f"Test R² (T):    {metrics['r2_T']:.4f}")
print(f"Test R² (bc_V): {metrics['r2_bc_V']:.4f}")
print(f"Checkpoint:     {summary['ckpt_dir']}")
print("=" * 60)